# 🚀 Модуль 2. Урок 4. Подзапросы и Common Table Expressions (CTE): читаемый модульный SQL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_02_sql/lesson_04_subqueries_and_cte.ipynb)

## 📌 Бизнес-контекст и методология
В промышленной аналитике CTE через `WITH` заменяют вложенные подзапросы, делая код модульным.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(ensure_dataset("ecommerce.db"))

query = """
WITH OrderTotals AS (
    SELECT 
        order_id,
        SUM(total_amount) AS order_total
    FROM order_items
    GROUP BY order_id
),
CustomerAverages AS (
    SELECT 
        o.customer_id,
        AVG(ot.order_total) AS customer_avg_order,
        COUNT(o.order_id) AS total_orders
    FROM orders o
    JOIN OrderTotals ot ON o.order_id = ot.order_id
    WHERE o.status = 'Completed'
    GROUP BY o.customer_id
)
SELECT 
    c.customer_id,
    c.first_name,
    c.city,
    ROUND(ca.customer_avg_order, 2) AS avg_check,
    ca.total_orders
FROM CustomerAverages ca
JOIN customers c ON ca.customer_id = c.customer_id
ORDER BY ca.customer_avg_order DESC
LIMIT 5;
"""

df_cte = pd.read_sql_query(query, conn)
df_cte


,customer_id,first_name,city,avg_check,total_orders
0,193,User_193,Samara,5987.44,1
1,1161,User_1161,Kazan,5783.32,1
2,791,User_791,Saint Petersburg,5723.74,1
3,1091,User_1091,Yekaterinburg,5063.08,1
4,40,User_40,Nizhny Novgorod,4973.79,1


### 🎯 Практическое задание 1: CTE для поиска первого и последнего заказа
Напишите запрос с CTE, находящим дату первого и последнего заказа клиента.

In [3]:
### ВАШ КОД ЗДЕСЬ
query_lifespan = """
WITH FirstOrders AS (
    SELECT 
        customer_id,
        MIN(order_date) AS first_order_date,
        MAX(order_date) AS last_order_date,
        COUNT(order_id) AS total_orders
    FROM orders
    WHERE status = 'Completed'
    GROUP BY customer_id
    HAVING COUNT(order_id) > 1
)
SELECT 
    customer_id,
    first_order_date,
    last_order_date,
    total_orders,
    ROUND(JULIANDAY(last_order_date) - JULIANDAY(first_order_date), 1) AS lifespan_days
FROM FirstOrders
ORDER BY lifespan_days DESC
LIMIT 10;
"""

df_life = pd.read_sql_query(query_lifespan, conn)
print(df_life.head(5))

assert not df_life.empty, "Список не должен быть пустым"
assert (df_life['lifespan_days'] >= 0).all(), "Срок жизни должен быть неотрицательным"
print("✅ Тест успешно пройден: CTE отработал корректно!")


   customer_id     first_order_date      last_order_date  total_orders  \
0         1022  2025-09-09 00:00:00  2026-06-12 00:00:00            11   
1         1067  2025-12-24 00:00:00  2026-09-24 00:00:00             9   
2         1110  2025-06-16 00:00:00  2026-03-09 00:00:00             9   
3          377  2025-07-20 00:00:00  2026-04-06 00:00:00            11   
4          869  2025-05-26 00:00:00  2026-02-01 00:00:00            11   

   lifespan_days  
0          276.0  
1          274.0  
2          266.0  
3          260.0  
4          251.0  
✅ Тест успешно пройден: CTE отработал корректно!


### 💡 Почему CTE эффективнее подзапросов?
1. Модульность и самодокументируемость.
2. Возможность многократного обращения к одному CTE.